<a href="https://colab.research.google.com/github/amnajamil381-png/AI-ML-Internship-Phase-II/blob/main/Task%201%3A%20News%20Topic%20Classifier%20Using%20BERT/%20News_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📰 News Classification using BERT

## Project Overview

This project fine-tunes the pretrained **BERT (bert-base-uncased)** model to classify news headlines into one of four categories using the **AG News** dataset.

The project demonstrates the complete NLP pipeline, including:

- Loading and exploring the dataset
- Text tokenization
- Fine-tuning a pretrained BERT model
- Model evaluation using Accuracy and F1 Score
- Saving the trained model
- Preparing the model for Streamlit deployment

## Install Required Libraries

Install the necessary Python libraries for loading the dataset, preprocessing text, training the model, and evaluating its performance.

In [1]:
!pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


Uninstalling the older versions of the libraries and then re-installing newrer versions to verify that the huggingface_hub works fine and loads the AG News Dataset

In [2]:
!pip uninstall -y transformers datasets huggingface_hub tokenizers accelerate

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: huggingface_hub 1.20.1
Uninstalling huggingface_hub-1.20.1:
  Successfully uninstalled huggingface_hub-1.20.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0


In [3]:
!pip install -q \
transformers==4.53.0 \
datasets==3.6.0 \
huggingface_hub==0.34.4 \
accelerate==1.8.1 \
evaluate \
scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.34.4 which is incompatible.


## Import Required Libraries

Import all the libraries needed for dataset handling, model training, evaluation, and prediction.

In [4]:
import torch # Deep Learning framework

from datasets import load_dataset #load datasets
# Hugging face transformers
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

# Evaluation metrics
from sklearn.metrics import accuracy_score, f1_score

## Load the AG News Dataset

The AG News dataset contains news headlines categorized into four classes:

- World
- Sports
- Business
- Sci/Tech

The dataset is automatically divided into training and testing sets.

In [5]:
dataset = load_dataset("ag_news")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Explore the Dataset

Let's inspect a few samples to understand the structure of the dataset.

Each record contains:

- News headline (text)
- Corresponding category label

In [25]:
# Display first five samples
dataset["train"][:5]

{'text': ["Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
  'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.',
  "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.",
  'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.',
  'Oil prices soar to all-time record, posing new menace to 

## Load the BERT Tokenizer

BERT cannot process raw text directly.

The tokenizer converts text into numerical tokens while adding the required special tokens used by the model.

In [6]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased") #Load the pretrained BERT Tokenizer

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

## Tokenize the Dataset

Convert every news headline into numerical input IDs and attention masks that can be understood by BERT.

Padding ensures equal sequence lengths, while truncation prevents excessively long inputs.

In [7]:
# Tokenization function
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

#Apply the tokenization on the dataset
tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Load the Pretrained BERT Model

Load the pretrained **bert-base-uncased** model and modify the classification head for four output classes.

In [8]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Define Evaluation Metrics

The model performance will be measured using:

- Accuracy
- Weighted F1 Score

In [9]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Function to compute evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": accuracy,
        "f1": f1,
    }

## Configure Training Parameters

Define the training configuration, including:

- Number of epochs
- Batch size
- Evaluation strategy
- Logging frequency

In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=100,
    report_to="none"      # Disable all experiment tracking
)

## Initialize the Trainer

The Hugging Face Trainer simplifies the training and evaluation process by managing optimization, validation, and logging.

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

trainer.train() # Fine-Tune the model

Step,Training Loss
100,0.543900
200,0.394600
300,0.367200
400,0.319300
500,0.323700
600,0.321400
700,0.308300
800,0.296500
900,0.309900
1000,0.261700


TrainOutput(global_step=22500, training_loss=0.16346808433532714, metrics={'train_runtime': 10597.4871, 'train_samples_per_second': 33.97, 'train_steps_per_second': 2.123, 'total_flos': 2.368042020864e+16, 'train_loss': 0.16346808433532714, 'epoch': 3.0})

## Evaluate Model Performance

Evaluate the trained model using the test dataset and report the Accuracy and Weighted F1 Score.

In [12]:
predictions = trainer.predict(tokenized_dataset["test"])

## Save the Trained Model

Save both the fine-tuned BERT model and tokenizer for future inference and deployment using Streamlit.

In [13]:
# Save the trained model
trainer.save_model("bert_news_classifier")

# Save the tokenizer
tokenizer.save_pretrained("bert_news_classifier")

('bert_news_classifier/tokenizer_config.json',
 'bert_news_classifier/special_tokens_map.json',
 'bert_news_classifier/vocab.txt',
 'bert_news_classifier/added_tokens.json')

## Test the Model on a Custom News Headline

Use the trained model to classify a user-defined news headline and display the predicted category along with the confidence score.

In [16]:
label_names = ["World", "Sports", "Business", "Sci/Tech"]

def predict_category(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=1).item()
    return label_names[predicted_class]

print(predict_category("Stock markets rally after interest rate cut"))
print(predict_category("Local team wins championship in overtime thriller"))
print(predict_category("New exoplanet discovered by astronomers"))
print(predict_category("UN Security Council meets over ceasefire talks"))

Business
Sports
Sci/Tech
World


# Project Summary

In this project, a pretrained **BERT (bert-base-uncased)** model was fine-tuned on the AG News dataset to classify news headlines into four categories.

### Workflow

- Loaded the AG News dataset
- Explored and analyzed the data
- Tokenized text using the BERT tokenizer
- Fine-tuned a pretrained BERT model
- Evaluated the model using Accuracy and Weighted F1 Score
- Saved the trained model for deployment
- Prepared the model for deployment with Streamlit and Hugging Face Spaces

This project demonstrates the practical application of **Natural Language Processing (NLP)**, **Transformer-based models**, and **Transfer Learning** for text classification.